# ECAPA-TDNN finetuning — improved (Common Voice PL)

Wersja ulepszona vs `ecapa_finetune_orig.ipynb`. Zachowana struktura, dodane następujące techniki:

**Tier A — szybkie wygrane (zero/near-zero compute cost):**
- Mixed Precision Training (AMP) — ~2× szybciej na A100
- BATCH_SIZE 64 → 192 (wykorzystanie A100 40 GB)
- AdamW + weight decay (regularizacja)
- Warmup (1 epoka) + cosine LR schedule
- Gradient clipping (norm=5.0)
- AAM margin curriculum (0.0 → 0.2 w 4 epokach)

**Tier B — strukturalne ulepszenia:**
- Speaker-balanced batch sampler (M=64 mówców × N=3 nagrania/batch)
- SEGMENT_SECONDS 3.0 → 4.0
- Sub-center AAM-Softmax (K=3 sub-centers/klasa)
- Multi-crop TTA przy ewaluacji (3 cropy uśrednione)
- AS-norm (Adaptive Score Normalization) na ewaluacji

**Tier C — zaawansowane augmentacje + LMFT:**
- Speed Perturbation (0.9× / 1.0× / 1.1×, offline preprocessing)
- SpecAugment intensification (3 maski, time_mask=35)
- Phase 3 — Large Margin Fine-Tuning (margin=0.5, segment=6s, 2 epoki)

Oczekiwane wyniki: dev EER **0.0452 → ~0.025–0.030** (orig vs improved).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

# Instalacja zależności (najnowsza wersja speechbrain rozwiązuje problemy z kompatybilnością)
!pip install -q speechbrain pandas tqdm scikit-learn matplotlib
!apt-get -qq install -y ffmpeg

## Konfiguracja

Wszystkie hiperparametry w jednym miejscu. Komentarze przy zmianach vs orig.

In [ ]:
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive")

# --- Ścieżki ---
TAR_GZ_PATH  = DRIVE_BASE / "biometrics/voice_authorization/data/cv-corpus-25.0-2026-03-09-pl.tar.gz"
RESULTS_DIR  = DRIVE_BASE / "biometrics/voice_authorization/results_improved"  # osobny folder

EXTRACT_DIR  = Path("/content/cv_pl_extracted")
WAV_DIR      = Path("/content/wav_16k")
PRETRAINED_DIR = Path("/content/pretrained_ecapa")

# --- Filtrowanie / podział ---
MIN_CLIPS_PER_SPEAKER = 10
MAX_CLIPS_PER_SPEAKER = 50
TRAIN_RATIO = 0.70
DEV_RATIO   = 0.15
TEST_RATIO  = 0.15

# --- Audio ---
TARGET_SR        = 16000
SEGMENT_SECONDS  = 4.0               # was 3.0 — dłuższe okno, lepszy embedding
EVAL_MAX_SECONDS = 8.0
EVAL_CROP_SECONDS = 4.0              # długość pojedynczego cropu przy TTA

# --- ECAPA / AAM-Softmax ---
EMBEDDING_SIZE     = 192
AAM_S              = 30.0
AAM_MARGIN_FINAL   = 0.2             # docelowy margines po curriculum
MARGIN_RAMP_EPOCHS = 4               # epok do osiągnięcia max marginesu (faza 2)
SUBCENTER_K        = 3               # sub-center AAM-Softmax (K=3 sub-centers/klasa)

# --- Trening (Tier A) ---
USE_AMP        = True                # mixed precision na A100
BATCH_SIZE     = 192                 # was 64 — wykorzystanie pamięci A100
NUM_WORKERS    = 4                   # was 2
PHASE1_LR      = 1e-4                # was 1e-3 — niższy, stabilniejszy
PHASE1_EPOCHS  = 5
PHASE2_LR      = 2e-5                # was 1e-5 — wyższy, kompensowany warmupem+cosine
PHASE2_EPOCHS  = 15
WEIGHT_DECAY   = 2e-5                # AdamW regularization
WARMUP_EPOCHS  = 1                   # liniowy warmup w fazie 2
GRAD_CLIP_NORM = 5.0

# --- Balanced sampler (Tier B) ---
BALANCED_SAMPLER_M = 64              # mówców per batch
BALANCED_SAMPLER_N = 3               # nagrań per mówca
# M*N == BATCH_SIZE = 192

# --- Augmentacje ---
AUG_NOISE_PROB     = 0.5
AUG_NOISE_SNR_DB   = (5, 20)
AUG_GAIN_DB        = 6.0
AUG_SPEED_PROB     = 0.6             # NEW: prob. użycia speed-perturbed wariantu
SPECAUG_FREQ_MASK  = 15
SPECAUG_TIME_MASK  = 35              # was 25
SPECAUG_NUM_MASKS  = 3               # was 2

# --- Ewaluacja ---
NUM_POSITIVE_PAIRS  = 3000
NUM_NEGATIVE_PAIRS  = 3000
EVAL_NUM_CROPS      = 3              # multi-crop TTA
ASNORM_COHORT_SIZE  = 500            # liczba nagrań w cohort
ASNORM_TOP_N        = 300            # top-N do statystyk normalizacji

# --- LMFT (faza 3) ---
LMFT_ENABLED         = True
LMFT_EPOCHS          = 2
LMFT_MARGIN          = 0.5
LMFT_SEGMENT_SECONDS = 6.0
LMFT_LR              = 5e-6
LMFT_BATCH_SIZE      = 96            # mniejszy ze względu na dłuższy segment
LMFT_SAMPLER_M       = 32            # 32 × 3 = 96
LMFT_SAMPLER_N       = 3

SEED = 42

In [ ]:
import os
import random
import shutil
import subprocess
import tarfile
import math
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Reprodukowalność
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"AMP enabled: {USE_AMP}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
WAV_DIR.mkdir(parents=True, exist_ok=True)

## Rozpakowanie korpusu

Idempotentne — jeśli `EXTRACT_DIR` już zawiera dane, pomijamy rozpakowanie.

In [ ]:
def find_corpus_root(base: Path) -> Path:
    for tsv in base.rglob("validated.tsv"):
        return tsv.parent
    raise FileNotFoundError(f"Nie znaleziono validated.tsv w {base}")

if not any(EXTRACT_DIR.iterdir()):
    print(f"Rozpakowywanie {TAR_GZ_PATH} → {EXTRACT_DIR} ...")
    with tarfile.open(TAR_GZ_PATH, "r:gz") as tf:
        tf.extractall(EXTRACT_DIR)
    print("Gotowe.")
else:
    print(f"EXTRACT_DIR już niepusty: pomijam rozpakowanie")

CORPUS_ROOT = find_corpus_root(EXTRACT_DIR)
CLIPS_DIR   = CORPUS_ROOT / "clips"
print(f"CORPUS_ROOT: {CORPUS_ROOT}")
print(f"CLIPS_DIR:   {CLIPS_DIR}  ({sum(1 for _ in CLIPS_DIR.iterdir())} plików)")

## Filtrowanie i open-set split po `client_id`

Identyczna logika jak w oryginale — ten sam `SEED=42` gwarantuje identyczne splity.

In [ ]:
df = pd.read_csv(CORPUS_ROOT / "validated.tsv", sep="\t", low_memory=False)
df = df[["client_id", "path"]].dropna()
print(f"Validated total: {len(df)} klipów, {df['client_id'].nunique()} unikalnych mówców")

counts = df.groupby("client_id").size()
ok_speakers = counts[counts >= MIN_CLIPS_PER_SPEAKER].index
df = df[df["client_id"].isin(ok_speakers)].copy()
print(f"Po filtrze ≥{MIN_CLIPS_PER_SPEAKER} klipów: {len(df)} klipów, {df['client_id'].nunique()} mówców")

df = (df.groupby("client_id", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), MAX_CLIPS_PER_SPEAKER), random_state=SEED)))
print(f"Po cap ≤{MAX_CLIPS_PER_SPEAKER} klipów/mówca: {len(df)} klipów, {df['client_id'].nunique()} mówców")

all_speakers = sorted(df["client_id"].unique())
train_spk, temp_spk = train_test_split(
    all_speakers, test_size=(DEV_RATIO + TEST_RATIO), random_state=SEED
)
dev_spk, test_spk = train_test_split(
    temp_spk, test_size=TEST_RATIO / (DEV_RATIO + TEST_RATIO), random_state=SEED
)
train_spk, dev_spk, test_spk = set(train_spk), set(dev_spk), set(test_spk)

train_id_to_label = {cid: i for i, cid in enumerate(sorted(train_spk))}
num_train_classes = len(train_id_to_label)

def make_records(speakers, with_label=False):
    rows = df[df["client_id"].isin(speakers)]
    if with_label:
        return [(r.path, train_id_to_label[r.client_id]) for r in rows.itertuples()]
    return [(r.path, r.client_id) for r in rows.itertuples()]

train_data = make_records(train_spk, with_label=True)
dev_data   = make_records(dev_spk)
test_data  = make_records(test_spk)

print()
print(f"Train: {len(train_spk)} mówców, {len(train_data)} klipów  ({num_train_classes} klas)")
print(f"Dev:   {len(dev_spk)} mówców, {len(dev_data)} klipów")
print(f"Test:  {len(test_spk)} mówców, {len(test_data)} klipów")

for name, spk in [("train_split.txt", train_spk), ("valid_split.txt", dev_spk), ("test_split.txt", test_spk)]:
    (RESULTS_DIR / name).write_text("\n".join(sorted(spk)) + "\n")
    print(f"Zapisano {name}: {len(spk)} mówców")

## Pre-konwersja MP3 → WAV 16 kHz mono

Identycznie jak w oryginale. Jednorazowe, idempotentne.

In [ ]:
def mp3_to_wav(mp3_path: Path, wav_path: Path) -> bool:
    if wav_path.exists():
        return True
    wav_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-i", str(mp3_path),
           "-ar", str(TARGET_SR), "-ac", "1",
           "-c:a", "pcm_s16le", str(wav_path)]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        return True
    except subprocess.CalledProcessError:
        return False

def wav_path_for(client_id: str, mp3_name: str) -> Path:
    return WAV_DIR / client_id / (Path(mp3_name).stem + ".wav")

needed = []
for spk_set in (train_spk, dev_spk, test_spk):
    for r in df[df["client_id"].isin(spk_set)].itertuples():
        mp3 = CLIPS_DIR / r.path
        wav = wav_path_for(r.client_id, r.path)
        needed.append((mp3, wav))

print(f"Klipów do konwersji: {len(needed)}")

failed = 0
with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
    futures = [ex.submit(mp3_to_wav, mp3, wav) for mp3, wav in needed]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="MP3→WAV"):
        if not fut.result():
            failed += 1

print(f"Konwersja zakończona. Błędów: {failed}")

In [ ]:
path_to_cid = dict(zip(df["path"], df["client_id"]))

def to_wav_records(records, with_label):
    out = []
    for mp3_name, key in records:
        cid = path_to_cid[mp3_name] if with_label else key
        wav = wav_path_for(cid, mp3_name)
        if wav.exists():
            out.append((str(wav), key))
    return out

train_data = to_wav_records(train_data, with_label=True)
dev_data   = to_wav_records(dev_data,   with_label=False)
test_data  = to_wav_records(test_data,  with_label=False)
print(f"Po filtracji do istniejących WAV-ów:")
print(f"  train: {len(train_data)}, dev: {len(dev_data)}, test: {len(test_data)}")

## Speed Perturbation (offline preprocessing, Tier C)

Dla każdego klipu treningowego generujemy 2 dodatkowe wersje: `_sp0.9.wav` i `_sp1.1.wav`
(0.9× i 1.1× tempo). Daje to efektywnie 3× więcej "klas akustycznych" — speed perturbation
tworzy syntetycznych mówców z lekko zmienioną wysokością tonu.

Idempotentne, równoległe. Robione tylko dla **train** (nie dev/test — żeby ewaluacja była uczciwa).

In [ ]:
SPEED_FACTORS = [0.9, 1.1]

def speed_variant_path(wav_path: Path, factor: float) -> Path:
    return wav_path.parent / f"{wav_path.stem}_sp{factor}.wav"

def speed_perturb(wav_in: Path, wav_out: Path, factor: float) -> bool:
    if wav_out.exists():
        return True
    wav_out.parent.mkdir(parents=True, exist_ok=True)
    # atempo filter: zachowuje pitch (resampling tempa)
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-i", str(wav_in),
           "-filter:a", f"atempo={factor}",
           "-ar", str(TARGET_SR), "-ac", "1",
           "-c:a", "pcm_s16le", str(wav_out)]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        return True
    except subprocess.CalledProcessError:
        return False

# Generujemy warianty tylko dla train_data
train_wavs = [Path(wp) for wp, _ in train_data]
print(f"Generuję {len(train_wavs)} × {len(SPEED_FACTORS)} = {len(train_wavs)*len(SPEED_FACTORS)} wariantów")

speed_failed = 0
tasks = []
for wp in train_wavs:
    for f in SPEED_FACTORS:
        tasks.append((wp, speed_variant_path(wp, f), f))

with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
    futures = [ex.submit(speed_perturb, wp, out, f) for (wp, out, f) in tasks]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="SpeedPerturb"):
        if not fut.result():
            speed_failed += 1

print(f"Speed perturbation gotowe. Błędów: {speed_failed}")

def get_speed_variants(wav_path_str: str) -> list:
    """Zwraca listę dostępnych wariantów (orig + sp0.9 + sp1.1)."""
    wp = Path(wav_path_str)
    variants = [str(wp)]
    for f in SPEED_FACTORS:
        sp = speed_variant_path(wp, f)
        if sp.exists():
            variants.append(str(sp))
    return variants

## Dataset + BalancedBatchSampler

- `CVPLSpeakerDataset` — wspiera speed perturbation (losowy wybór wariantu)
- `BalancedBatchSampler` — wymusza M mówców × N nagrań/batch (kluczowe dla AAM-Softmax)

In [ ]:
class CVPLSpeakerDataset(Dataset):
    def __init__(self, data, segment_seconds: float, mode: str = "train",
                 use_speed_perturb: bool = False, speed_prob: float = AUG_SPEED_PROB):
        self.data = data
        self.segment_len = int(segment_seconds * TARGET_SR)
        self.mode = mode
        self.use_speed = use_speed_perturb
        self.speed_prob = speed_prob
        # Pre-cache speed variants (tylko dla train)
        if self.use_speed:
            self.variants = {path: get_speed_variants(path) for path, _ in data}
        else:
            self.variants = {}

    def __len__(self):
        return len(self.data)

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        return wav.mean(0)

    def _crop_or_pad(self, wav):
        T = wav.shape[0]
        if T >= self.segment_len:
            if self.mode == "train":
                start = random.randint(0, T - self.segment_len)
            else:
                start = max(0, (T - self.segment_len) // 2)
            return wav[start : start + self.segment_len]
        pad = self.segment_len - T
        return F.pad(wav, (0, pad))

    def _select_path(self, path):
        if not self.use_speed or random.random() >= self.speed_prob:
            return path
        return random.choice(self.variants.get(path, [path]))

    def __getitem__(self, idx):
        path, label = self.data[idx]
        path = self._select_path(path)
        wav = self._load(path)
        wav = self._crop_or_pad(wav)
        if self.mode == "train":
            wav = self._augment_waveform(wav)
        return wav, label

    def _augment_waveform(self, wav):
        # Random gain
        gain_db = random.uniform(-AUG_GAIN_DB, AUG_GAIN_DB)
        wav = wav * (10.0 ** (gain_db / 20.0))
        # Gaussian noise
        if random.random() < AUG_NOISE_PROB:
            snr_db = random.uniform(*AUG_NOISE_SNR_DB)
            sig_power = wav.pow(2).mean().clamp(min=1e-10)
            noise = torch.randn_like(wav)
            noise_power = sig_power / (10.0 ** (snr_db / 10.0))
            noise = noise * torch.sqrt(noise_power / noise.pow(2).mean().clamp(min=1e-10))
            wav = wav + noise
        return wav


class BalancedBatchSampler(Sampler):
    """Każdy batch: M mówców × N nagrań/mówcę. Wymusza obecność klas pozytywnych w batchu."""
    def __init__(self, labels, M: int, N: int, num_batches=None, seed: int = SEED):
        self.labels = list(labels)
        self.M = M
        self.N = N
        self.label_to_indices = defaultdict(list)
        for i, lab in enumerate(self.labels):
            self.label_to_indices[lab].append(i)
        self.unique_labels = list(self.label_to_indices.keys())
        if num_batches is None:
            total = sum(len(v) for v in self.label_to_indices.values())
            self.num_batches = max(1, total // (M * N))
        else:
            self.num_batches = num_batches
        self.rng = random.Random(seed)

    def __iter__(self):
        for _ in range(self.num_batches):
            speakers = self.rng.sample(self.unique_labels, self.M)
            batch = []
            for spk in speakers:
                indices = self.label_to_indices[spk]
                if len(indices) >= self.N:
                    batch.extend(self.rng.sample(indices, self.N))
                else:
                    batch.extend(self.rng.choices(indices, k=self.N))
            yield batch

    def __len__(self):
        return self.num_batches


# Train loader (faza 1 + faza 2)
train_dataset = CVPLSpeakerDataset(train_data, SEGMENT_SECONDS, mode="train", use_speed_perturb=True)
train_labels = [lab for _, lab in train_data]
train_sampler = BalancedBatchSampler(train_labels, M=BALANCED_SAMPLER_M, N=BALANCED_SAMPLER_N)
train_loader  = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
print(f"Train dataset: {len(train_dataset)} próbek, batches/epoch: {len(train_loader)} (M={BALANCED_SAMPLER_M}×N={BALANCED_SAMPLER_N}={BALANCED_SAMPLER_M*BALANCED_SAMPLER_N})")

# Sanity check pierwszego batcha
sample_wavs, sample_labels = next(iter(train_loader))
assert sample_wavs.shape == (BALANCED_SAMPLER_M * BALANCED_SAMPLER_N, int(SEGMENT_SECONDS * TARGET_SR)), \
    f"Batch shape: {sample_wavs.shape}"
unique_in_batch = len(set(sample_labels.tolist()))
print(f"Sanity: batch.shape={sample_wavs.shape}, unikalnych mówców w batchu: {unique_in_batch}/{BALANCED_SAMPLER_M}")

## Model: ECAPA-TDNN backbone + Sub-Center AAM-Softmax

- Pretrained ECAPA z SpeechBrain (192-D, ~20.7M params)
- `SubCenterArcMarginProduct(K=3)` — robust to noisy labels (Common Voice ma różne warunki nagrania per mówca)
- `set_margin()` umożliwia curriculum learning

In [ ]:
import huggingface_hub
import huggingface_hub.file_download
from speechbrain.inference.speaker import EncoderClassifier

# Tworzymy pusty custom.py w razie gdyby biblioteka go wymagała
with open("custom.py", "w") as f:
    pass

# Bezpieczny patch huggingface_hub (raz)
if not hasattr(huggingface_hub, '_orig_hf_hub_download_clean'):
    huggingface_hub._orig_hf_hub_download_clean = huggingface_hub.file_download.hf_hub_download

def _patched_hf_hub_download(*args, **kwargs):
    kwargs.pop("use_auth_token", None)
    filename = kwargs.get("filename") or (args[1] if len(args) > 1 else None)
    try:
        return huggingface_hub._orig_hf_hub_download_clean(*args, **kwargs)
    except Exception as e:
        if type(e).__name__ in ["RemoteEntryNotFoundError", "EntryNotFoundError", "HTTPStatusError"]:
            if filename == "custom.py":
                return os.path.abspath("custom.py")
        raise

huggingface_hub.hf_hub_download = _patched_hf_hub_download
huggingface_hub.file_download.hf_hub_download = _patched_hf_hub_download

if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile']

spk_module = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=str(PRETRAINED_DIR),
    run_opts={"device": str(device)},
)
compute_features = spk_module.mods.compute_features
mean_var_norm    = spk_module.mods.mean_var_norm
embedding_model  = spk_module.mods.embedding_model
embedding_model  = embedding_model.to(device)

# SpecAugment
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=SPECAUG_FREQ_MASK).to(device)
time_mask = torchaudio.transforms.TimeMasking(time_mask_param=SPECAUG_TIME_MASK).to(device)

def forward_embedding(wavs: torch.Tensor, lengths: torch.Tensor, training: bool):
    """wavs: (B, T). Zwraca (B, EMBEDDING_SIZE) L2-normalized."""
    feats = compute_features(wavs)
    feats = mean_var_norm(feats, lengths)
    if training:
        feats_t = feats.transpose(1, 2)
        for _ in range(SPECAUG_NUM_MASKS):
            feats_t = freq_mask(feats_t)
            feats_t = time_mask(feats_t)
        feats = feats_t.transpose(1, 2)
    emb = embedding_model(feats)
    emb = emb.squeeze(1)
    return F.normalize(emb, p=2, dim=-1)

In [ ]:
class SubCenterArcMarginProduct(nn.Module):
    """Sub-center AAM-Softmax (K sub-centers/klasa).

    Layout weight: (out_features * K, in_features), gdzie weight[k*N : (k+1)*N]
    to k-te sub-centers dla wszystkich N klas. Max po dim K daje finalny cosine.
    """
    def __init__(self, in_features, out_features, K=3, s=30.0, m=0.0):
        super().__init__()
        self.s = s
        self.m = m
        self.K = K
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(out_features * K, in_features))
        nn.init.xavier_uniform_(self.weight)
        self._update_margin_constants()

    def _update_margin_constants(self):
        self.cos_m = math.cos(self.m)
        self.sin_m = math.sin(self.m)
        self.th = math.cos(math.pi - self.m)
        self.mm = math.sin(math.pi - self.m) * self.m

    def set_margin(self, m: float):
        self.m = m
        self._update_margin_constants()

    def forward(self, embedding, label):
        # Pełna macierz cosine (B, K * out_features)
        cosine_all = F.linear(F.normalize(embedding), F.normalize(self.weight))
        B = cosine_all.shape[0]
        cosine_all = cosine_all.view(B, self.K, self.out_features)
        # Max po sub-centerach
        cosine, _ = cosine_all.max(dim=1)
        # ArcFace transformation
        if self.m > 0:
            sine = torch.sqrt((1.0 - cosine.pow(2)).clamp(0, 1))
            phi = cosine * self.cos_m - sine * self.sin_m
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        else:
            phi = cosine
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output


head = SubCenterArcMarginProduct(EMBEDDING_SIZE, num_train_classes, K=SUBCENTER_K, s=AAM_S, m=0.0).to(device)
criterion = nn.CrossEntropyLoss()

print(f"Backbone (ECAPA) parametry: {sum(p.numel() for p in embedding_model.parameters()):,}")
print(f"Head (SubCenter K={SUBCENTER_K}): {sum(p.numel() for p in head.parameters()):,}")

## Generowanie par i metryki (TTA + AS-norm)

- `generate_pairs` — 1:1 z oryginału
- `extract_embeddings` — multi-crop TTA (3 cropy, uśrednione, renormalizowane)
- `compute_metrics` — 1:1 z oryginału
- `compute_asnorm_metrics` — AS-norm score normalization

In [ ]:
def generate_pairs(data, num_positive=NUM_POSITIVE_PAIRS, num_negative=NUM_NEGATIVE_PAIRS, seed=SEED):
    rng = random.Random(seed)
    id_to_files = defaultdict(list)
    for f, cid in data:
        id_to_files[cid].append(f)
    pos_ids = [cid for cid, fs in id_to_files.items() if len(fs) >= 2]
    all_ids = list(id_to_files.keys())

    pairs = []
    for _ in range(num_positive):
        cid = rng.choice(pos_ids)
        f1, f2 = rng.sample(id_to_files[cid], 2)
        pairs.append((f1, f2, 1))
    for _ in range(num_negative):
        c1, c2 = rng.sample(all_ids, 2)
        f1 = rng.choice(id_to_files[c1])
        f2 = rng.choice(id_to_files[c2])
        pairs.append((f1, f2, 0))
    rng.shuffle(pairs)
    return pairs


@torch.no_grad()
def extract_embeddings(file_list_or_pairs, max_seconds: float = EVAL_MAX_SECONDS,
                        crop_seconds: float = EVAL_CROP_SECONDS,
                        num_crops: int = EVAL_NUM_CROPS, batch_size: int = 32):
    """Multi-crop TTA: każdy plik → num_crops cropów → uśredniony L2-normalized embedding.

    file_list_or_pairs: lista plików lub lista par (każda para = (f1, f2, label)).
    """
    embedding_model.eval()
    # Wyciągnij unikalne pliki — robust detection: tuple/list = pary, string = lista plików
    if file_list_or_pairs and isinstance(file_list_or_pairs[0], (tuple, list)) and len(file_list_or_pairs[0]) == 3:
        unique_files = sorted({f for p in file_list_or_pairs for f in (p[0], p[1])})
    else:
        unique_files = sorted(set(file_list_or_pairs))

    max_len  = int(max_seconds * TARGET_SR)
    crop_len = int(crop_seconds * TARGET_SR)
    embeddings = {}

    def load_clip(path):
        wav, sr = torchaudio.load(path)
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        wav = wav.mean(0)
        # Cap przy max_len, żeby długie nagrania nie spowalniały
        if wav.shape[0] > max_len:
            start = max(0, (wav.shape[0] - max_len) // 2)
            wav = wav[start : start + max_len]
        return wav

    def get_crops(wav, n_crops, length):
        T = wav.shape[0]
        if T <= length:
            padded = F.pad(wav, (0, length - T))
            return [padded] * n_crops
        if n_crops == 1:
            start = max(0, (T - length) // 2)
            return [wav[start : start + length]]
        starts = np.linspace(0, T - length, n_crops).astype(int)
        return [wav[s : s + length] for s in starts]

    for i in range(0, len(unique_files), batch_size):
        batch_files = unique_files[i : i + batch_size]
        wavs = [load_clip(f) for f in batch_files]
        # n_crops × batch_size cropów, wszystkie o crop_len
        all_crops = []
        for w in wavs:
            crops = get_crops(w, num_crops, crop_len)
            # ujednolić długość (powinny już być crop_len, ale safety)
            crops_fixed = []
            for c in crops:
                if c.shape[0] < crop_len:
                    c = F.pad(c, (0, crop_len - c.shape[0]))
                elif c.shape[0] > crop_len:
                    c = c[:crop_len]
                crops_fixed.append(c)
            all_crops.append(crops_fixed)
        flat = [c for crops in all_crops for c in crops]
        padded = torch.stack(flat).to(device)
        lengths = torch.ones(padded.shape[0], device=device)
        emb_flat = forward_embedding(padded, lengths, training=False)
        emb_reshaped = emb_flat.view(len(batch_files), num_crops, -1)
        emb_avg = emb_reshaped.mean(dim=1)
        emb_avg = F.normalize(emb_avg, p=2, dim=-1)
        for j, f in enumerate(batch_files):
            embeddings[f] = emb_avg[j].cpu()
    return embeddings


def compute_metrics(pairs, embeddings):
    sims, labels = [], []
    for f1, f2, is_same in pairs:
        s = F.cosine_similarity(embeddings[f1].unsqueeze(0), embeddings[f2].unsqueeze(0)).item()
        sims.append(s); labels.append(is_same)
    sims = np.array(sims); labels = np.array(labels)
    fpr, tpr, thr = roc_curve(labels, sims)
    fnr = 1.0 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
    return {"eer": eer, "auc": auc(fpr, tpr), "fpr": fpr, "tpr": tpr, "fnr": fnr,
            "thresholds": thr, "similarities": sims, "labels": labels}


def compute_asnorm_metrics(pairs, embeddings, cohort_embeddings, top_n: int = ASNORM_TOP_N):
    """Adaptive Score Normalization (AS-norm).

    Dla pary (e1, e2) i score s = cos(e1, e2):
      m1, sd1 = top_n cohort sims do e1 → mean, std
      m2, sd2 = top_n cohort sims do e2 → mean, std
      s_norm = 0.5 * ((s - m1)/sd1 + (s - m2)/sd2)
    """
    cohort = torch.stack(list(cohort_embeddings.values()))
    cohort = F.normalize(cohort, p=2, dim=-1).to(device)
    top_n = min(top_n, cohort.shape[0])

    # Statystyki cohort dla każdego unikalnego pliku (cache)
    unique_files = sorted({f for p in pairs for f in (p[0], p[1])})
    stats = {}
    with torch.no_grad():
        for f in unique_files:
            e = F.normalize(embeddings[f].to(device), p=2, dim=-1).unsqueeze(0)
            sims_c = (cohort @ e.t()).squeeze(-1)
            topk_vals = torch.topk(sims_c, k=top_n).values
            stats[f] = (topk_vals.mean().item(), topk_vals.std().item() + 1e-6)

    sims_norm, labels = [], []
    for f1, f2, is_same in pairs:
        s = F.cosine_similarity(embeddings[f1].unsqueeze(0), embeddings[f2].unsqueeze(0)).item()
        m1, sd1 = stats[f1]
        m2, sd2 = stats[f2]
        s_n = 0.5 * ((s - m1) / sd1 + (s - m2) / sd2)
        sims_norm.append(s_n)
        labels.append(is_same)
    sims_norm = np.array(sims_norm); labels = np.array(labels)
    fpr, tpr, thr = roc_curve(labels, sims_norm)
    fnr = 1.0 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
    return {"eer": eer, "auc": auc(fpr, tpr), "fpr": fpr, "tpr": tpr, "fnr": fnr,
            "thresholds": thr, "similarities": sims_norm, "labels": labels}

## Baseline: EER pretrained na dev (przed douczaniem)

Z multi-crop TTA. Pokazujemy też single-crop (jak w orig) dla referencji.

In [ ]:
dev_pairs = generate_pairs(dev_data, NUM_POSITIVE_PAIRS, NUM_NEGATIVE_PAIRS, seed=SEED)
print(f"Wygenerowano {len(dev_pairs)} par dev")

# Single crop (jak w orig, dla porównania)
dev_emb_baseline_raw = extract_embeddings(dev_pairs, num_crops=1)
baseline_metrics_raw = compute_metrics(dev_pairs, dev_emb_baseline_raw)

# Multi-crop TTA
dev_emb_baseline_tta = extract_embeddings(dev_pairs, num_crops=EVAL_NUM_CROPS)
baseline_metrics_tta = compute_metrics(dev_pairs, dev_emb_baseline_tta)

print(f"BASELINE (pretrained, brak douczania):")
print(f"  Single crop   : EER={baseline_metrics_raw['eer']:.4f}  AUC={baseline_metrics_raw['auc']:.4f}")
print(f"  Multi-crop TTA: EER={baseline_metrics_tta['eer']:.4f}  AUC={baseline_metrics_tta['auc']:.4f}")

## Funkcja treningowa (AMP + grad clip + dynamic margin)

In [ ]:
def train_epoch(loader, optimizer, head, train_embedder: bool,
                scaler=None, current_margin: float = None):
    """Pojedyncza epoka. train_embedder=False → linear probing (zamrożony backbone).

    AMP via GradScaler jeśli scaler != None.
    current_margin nadpisuje margin głowicy przed forward.
    """
    if train_embedder:
        embedding_model.train()
    else:
        embedding_model.eval()
    head.train()

    if current_margin is not None:
        head.set_margin(current_margin)

    use_amp = scaler is not None and scaler.is_enabled()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    # Parametry trenowalne (do grad clip)
    trainable_params = [p for p in (list(embedding_model.parameters()) + list(head.parameters())) if p.requires_grad]

    for wavs, labels in tqdm(loader, leave=False):
        wavs = wavs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()
        lengths = torch.ones(wavs.shape[0], device=device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            if train_embedder:
                emb = forward_embedding(wavs, lengths, training=True)
            else:
                with torch.no_grad():
                    emb = forward_embedding(wavs, lengths, training=True)
            logits = head(emb, labels)
            loss = criterion(logits, labels)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP_NORM)
            optimizer.step()

        bs = wavs.shape[0]
        total_loss   += loss.item() * bs
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += bs

    return total_loss / total_samples, total_correct / total_samples

## Faza 1 — Linear probing (zamrożony backbone, margin=0.0)

AdamW, niższy LR (1e-4) niż w orig (1e-3), bo używamy sub-center K=3 i AdamW (większa regularizacja).
Margines = 0.0 → czysty softmax. Pozwala głowicy nauczyć się klas bez "shock" z AAM marginesem.

In [ ]:
# Zamrożenie embeddera
for p in embedding_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(head.parameters(), lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

print("=" * 60)
print(f"Faza 1: Linear Probing  |  lr={PHASE1_LR}, epoki={PHASE1_EPOCHS}, margin=0.0")
print("=" * 60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    loss, acc = train_epoch(train_loader, optimizer, head,
                            train_embedder=False, scaler=scaler, current_margin=0.0)
    print(f"Epoka {epoch}/{PHASE1_EPOCHS}  Loss: {loss:.4f}  Acc: {acc:.4f}")

## Faza 2 — Full fine-tuning (warmup + cosine + curriculum margin)

- AdamW + weight decay
- Warmup 1 epoka (liniowy 0 → PHASE2_LR), potem cosine annealing
- AAM margin: ramp 0.0 → AAM_MARGIN_FINAL (=0.2) liniowo w `MARGIN_RAMP_EPOCHS` (=4)
- Eval po każdej epoce (TTA), zapis best na podstawie dev EER

In [ ]:
# Odmrożenie embeddera
for p in embedding_model.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    list(embedding_model.parameters()) + list(head.parameters()),
    lr=PHASE2_LR,
    weight_decay=WEIGHT_DECAY,
)

# Warmup + cosine schedule (LambdaLR)
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, PHASE2_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_eer = float("inf")
eer_log_lines = [
    f"Baseline (pretrained, single crop): Dev EER={baseline_metrics_raw['eer']:.4f}  AUC={baseline_metrics_raw['auc']:.4f}",
    f"Baseline (pretrained, TTA):         Dev EER={baseline_metrics_tta['eer']:.4f}  AUC={baseline_metrics_tta['auc']:.4f}",
]

print("=" * 60)
print(f"Faza 2: Full Fine-tuning  |  lr={PHASE2_LR}, warmup+cosine, epoki={PHASE2_EPOCHS}")
print(f"Curriculum margin: 0 → {AAM_MARGIN_FINAL} w {MARGIN_RAMP_EPOCHS} epokach")
print("=" * 60)

for epoch in range(1, PHASE2_EPOCHS + 1):
    # Curriculum margin
    if epoch <= MARGIN_RAMP_EPOCHS:
        current_margin = AAM_MARGIN_FINAL * (epoch / MARGIN_RAMP_EPOCHS)
    else:
        current_margin = AAM_MARGIN_FINAL

    loss, acc = train_epoch(train_loader, optimizer, head,
                            train_embedder=True, scaler=scaler,
                            current_margin=current_margin)
    scheduler.step()

    # Eval — multi-crop TTA
    dev_emb = extract_embeddings(dev_pairs, num_crops=EVAL_NUM_CROPS)
    m = compute_metrics(dev_pairs, dev_emb)
    current_lr = scheduler.get_last_lr()[0]
    line = (f"Epoka {epoch:02d}/{PHASE2_EPOCHS}  m={current_margin:.3f}  lr={current_lr:.2e}  "
            f"Loss: {loss:.4f}  Acc: {acc:.4f}  Dev EER: {m['eer']:.4f}  AUC: {m['auc']:.4f}")
    eer_log_lines.append(line)
    print(line)

    if m["eer"] < best_eer:
        best_eer = m["eer"]
        torch.save(embedding_model.state_dict(), RESULTS_DIR / "ecapa_cv_pl_best.pth")
        print(f"  → nowy najlepszy (EER={m['eer']:.4f}) zapisany")

torch.save(embedding_model.state_dict(), RESULTS_DIR / "ecapa_cv_pl_phase2_final.pth")
print()
print(f"Najlepszy po fazie 2: EER={best_eer:.4f}")

## Faza 3 — Large Margin Fine-Tuning (LMFT)

Ostatnia faza z agresywnym marginesem (0.5) i długim segmentem (6s). Inspirowane VoxCeleb winners.
- `BATCH_SIZE` zmniejszony do 96 (32 mówców × 3 nagrania), żeby zmieścić 6s segment + AMP w 40 GB.
- Niski LR (5e-6) — drobna korekta wag, nie rewolucja.

In [ ]:
if LMFT_ENABLED:
    print("=" * 60)
    print(f"Faza 3: LMFT  |  margin={LMFT_MARGIN}, segment={LMFT_SEGMENT_SECONDS}s, lr={LMFT_LR}, batch={LMFT_BATCH_SIZE}")
    print("=" * 60)

    # Nowy dataset z dłuższym segmentem
    train_dataset_lmft = CVPLSpeakerDataset(
        train_data, LMFT_SEGMENT_SECONDS, mode="train", use_speed_perturb=True
    )
    sampler_lmft = BalancedBatchSampler(
        labels=train_labels, M=LMFT_SAMPLER_M, N=LMFT_SAMPLER_N, seed=SEED + 100
    )
    train_loader_lmft = DataLoader(
        train_dataset_lmft,
        batch_sampler=sampler_lmft,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    optimizer = torch.optim.AdamW(
        list(embedding_model.parameters()) + list(head.parameters()),
        lr=LMFT_LR,
        weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    for epoch in range(1, LMFT_EPOCHS + 1):
        loss, acc = train_epoch(train_loader_lmft, optimizer, head,
                                train_embedder=True, scaler=scaler,
                                current_margin=LMFT_MARGIN)
        dev_emb = extract_embeddings(dev_pairs, num_crops=EVAL_NUM_CROPS)
        m = compute_metrics(dev_pairs, dev_emb)
        line = (f"LMFT Epoka {epoch}/{LMFT_EPOCHS}  m={LMFT_MARGIN}  "
                f"Loss: {loss:.4f}  Acc: {acc:.4f}  Dev EER: {m['eer']:.4f}  AUC: {m['auc']:.4f}")
        eer_log_lines.append(line)
        print(line)
        if m["eer"] < best_eer:
            best_eer = m["eer"]
            torch.save(embedding_model.state_dict(), RESULTS_DIR / "ecapa_cv_pl_best.pth")
            print(f"  → nowy najlepszy (EER={m['eer']:.4f}) zapisany")

    torch.save(embedding_model.state_dict(), RESULTS_DIR / "ecapa_cv_pl_lmft_final.pth")

(RESULTS_DIR / "eer.log").write_text("\n".join(eer_log_lines) + "\n")
print()
print(f"Najlepszy model:  {RESULTS_DIR / 'ecapa_cv_pl_best.pth'}  (EER={best_eer:.4f})")
print(f"Logi:             {RESULTS_DIR / 'eer.log'}")

## Ewaluacja końcowa na zbiorze testowym

Wczytujemy najlepszy checkpoint i raportujemy trzy warianty:
1. Single crop (jak w orig — fair comparison)
2. Multi-crop TTA (3 cropy uśrednione)
3. TTA + AS-norm (cohort 500 nagrań z train)

In [ ]:
embedding_model.load_state_dict(torch.load(RESULTS_DIR / "ecapa_cv_pl_best.pth", map_location=device))
embedding_model.eval()

test_pairs = generate_pairs(test_data, NUM_POSITIVE_PAIRS, NUM_NEGATIVE_PAIRS, seed=SEED + 1)

# Embeddingi — raw vs TTA
test_emb_raw = extract_embeddings(test_pairs, num_crops=1)
test_emb_tta = extract_embeddings(test_pairs, num_crops=EVAL_NUM_CROPS)

# AS-norm cohort: losowe nagrania z train, embedding TTA
rng_cohort = random.Random(SEED + 2)
cohort_files = rng_cohort.sample([wp for wp, _ in train_data], ASNORM_COHORT_SIZE)
cohort_embeddings = extract_embeddings(cohort_files, num_crops=EVAL_NUM_CROPS)
print(f"Cohort do AS-norm: {len(cohort_embeddings)} nagrań")

# 3 warianty raportów
m_raw  = compute_metrics(test_pairs, test_emb_raw)
m_tta  = compute_metrics(test_pairs, test_emb_tta)
m_full = compute_asnorm_metrics(test_pairs, test_emb_tta, cohort_embeddings, top_n=ASNORM_TOP_N)

print("=" * 60)
print("Ewaluacja na zbiorze testowym (najlepszy model)")
print("=" * 60)
print(f"{'Wariant':<30} {'EER':>8}  {'AUC':>8}")
print(f"{'-'*30} {'-'*8}  {'-'*8}")
print(f"{'Single crop (jak orig)':<30} {m_raw['eer']:>8.4f}  {m_raw['auc']:>8.4f}")
print(f"{'Multi-crop TTA':<30} {m_tta['eer']:>8.4f}  {m_tta['auc']:>8.4f}")
print(f"{'TTA + AS-norm':<30} {m_full['eer']:>8.4f}  {m_full['auc']:>8.4f}")
print()

# Operating points (na najlepszym wariancie — TTA + AS-norm)
for target_fnr in [0.01, 0.05, 0.10]:
    idx = np.argmin(np.abs(m_full["fnr"] - target_fnr))
    print(f"FPR @ FNR={target_fnr:.2f}:  {m_full['fpr'][idx]:.4f}")
print()
for target_fpr in [0.01, 0.05, 0.10]:
    idx = np.argmin(np.abs(m_full["fpr"] - target_fpr))
    print(f"FNR @ FPR={target_fpr:.2f}:  {m_full['fnr'][idx]:.4f}")

# Δ vs baseline
print()
print(f"Δ vs baseline (single crop): EER {baseline_metrics_raw['eer']:.4f} → {m_raw['eer']:.4f}  "
      f"({(baseline_metrics_raw['eer'] - m_raw['eer']) / baseline_metrics_raw['eer'] * 100:+.1f}%)")
print(f"Δ vs baseline (TTA+AS-norm vs single-crop baseline): "
      f"{baseline_metrics_raw['eer']:.4f} → {m_full['eer']:.4f}  "
      f"({(baseline_metrics_raw['eer'] - m_full['eer']) / baseline_metrics_raw['eer'] * 100:+.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC porównawcze: 3 warianty
for m_name, m_data, color in [
    ("Raw", m_raw, "tab:blue"),
    ("TTA", m_tta, "tab:orange"),
    ("TTA+AS-norm", m_full, "tab:green"),
]:
    axes[0].plot(m_data["fpr"], m_data["tpr"], linewidth=2,
                 label=f"{m_name}: AUC={m_data['auc']:.4f}, EER={m_data['eer']:.4f}",
                 color=color)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve (test) — 3 warianty oceny")
axes[0].legend(loc="lower right"); axes[0].grid(True, alpha=0.3)

# Histogram cosine similarities (TTA+AS-norm)
pos = m_full["similarities"][m_full["labels"] == 1]
neg = m_full["similarities"][m_full["labels"] == 0]
eer_thr = m_full["thresholds"][np.argmin(np.abs(m_full["fpr"] - m_full["fnr"]))]
axes[1].hist(pos, bins=50, alpha=0.5, label="Positive pairs", color="green")
axes[1].hist(neg, bins=50, alpha=0.5, label="Negative pairs", color="red")
axes[1].axvline(x=eer_thr, color="blue", linestyle="--", label=f"EER thr = {eer_thr:.3f}")
axes[1].set_xlabel("Normalized score (AS-norm)"); axes[1].set_ylabel("Count")
axes[1].set_title("Score distribution (test, TTA+AS-norm)"); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Zapisano: {RESULTS_DIR / 'evaluation_plots.png'}")

## Eksport: `extract_embedding(wav_path) → np.ndarray[192]`

Funkcja gotowa do reuse w module enrollment/verification.
Stosuje multi-crop TTA dla pojedynczego pliku (3 cropy uśrednione).

In [ ]:
@torch.no_grad()
def extract_embedding(audio_path, use_tta: bool = True) -> np.ndarray:
    """L2-normalized 192-D embedding mówcy. use_tta=True → 3-crop average."""
    embedding_model.eval()
    wav, sr = torchaudio.load(str(audio_path))
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav.mean(0)

    crop_len = int(EVAL_CROP_SECONDS * TARGET_SR)
    n_crops = EVAL_NUM_CROPS if use_tta else 1

    T = wav.shape[0]
    if T <= crop_len:
        wav_p = F.pad(wav, (0, crop_len - T))
        crops = [wav_p] * n_crops
    elif n_crops == 1:
        start = max(0, (T - crop_len) // 2)
        crops = [wav[start : start + crop_len]]
    else:
        starts = np.linspace(0, T - crop_len, n_crops).astype(int)
        crops = [wav[s : s + crop_len] for s in starts]

    batch = torch.stack(crops).to(device)
    lengths = torch.ones(batch.shape[0], device=device)
    emb = forward_embedding(batch, lengths, training=False)
    emb_avg = F.normalize(emb.mean(dim=0, keepdim=True), p=2, dim=-1)
    return emb_avg.squeeze(0).cpu().numpy()

# Smoke test
example = next(WAV_DIR.rglob("*.wav"))
e = extract_embedding(example, use_tta=True)
print(f"Plik: {example.name}")
print(f"Embedding shape: {e.shape}, L2 norm: {np.linalg.norm(e):.4f}")